# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library following the Croissant specification.

### Dataset Source
The dataset source is provided via a Croissant schema JSON-LD URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. The Croissant schema uses the `@id` attribute to uniquely identify all data entities.

In [ ]:
# List available record sets with their @id, name, and description
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record sets:")
for rs in record_sets:
    print(f"- @id: {rs.id}\n  Name: {rs.name}\n  Description: {getattr(rs, 'description', '')}\n")
    if hasattr(rs, 'fields'):
        print("  Fields (by @id, name):")
        for field in rs.fields:
            print(f"    - @id: {field.id}, name: {field.name}")


## 3. Data Extraction
Load data from each record set into a `pandas.DataFrame` for analysis. Reference all entities (record sets, fields) by their `@id`.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        print(f"Loaded {len(records)} records from record set: {record_set_id}")
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
    except Exception as e:
        print(f"Error loading records from {record_set_id}: {e}")

# Display columns of the first record set as an example
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"\nColumns in record set {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply data filtering, normalization, and grouping using record set and field `@id`s.

In [ ]:
# Choose a record set and numeric field by their @id
if record_set_ids:
    # Select the first record set for this example
    selected_record_set_id = record_set_ids[0]
    df = dataframes[selected_record_set_id]
    print(f"Working with record set @id: {selected_record_set_id}")
    # Attempt to find a numeric field
    from pandas.api.types import is_numeric_dtype
    numeric_fields = [col for col in df.columns if is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean()
        # Filter records above the mean as an illustration
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (mean value):")
        display(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Group by another (categorical) field, if available
        possible_group_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
        if possible_group_fields:
            group_field = possible_group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped average of '{numeric_field}' by '{group_field}':")
            display(grouped_df.head())
    else:
        print("No numeric field found to demonstrate filtering and normalization.")

## 5. Visualization
Let's visualize the distribution of one numeric field and its relationship with a categorical field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_fields:
    # Histogram of the selected numeric field
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    # If grouping field exists, display boxplot
    if possible_group_fields:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=df[possible_group_fields[0]], y=df[numeric_field])
        plt.title(f"{numeric_field} by {possible_group_fields[0]}")
        plt.xlabel(possible_group_fields[0])
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We've demonstrated how to load, view, and process the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) using the `mlcroissant` Python library. The schema's record sets and fields can be explored dynamically using their `@id` attributes, ensuring robust data handling and reproducibility. For more advanced analysis, explore additional record sets and fields as documented in the Croissant metadata.